# Predictive Modeling Using Machine Learning
## Predicting Customer Churn with Supervised Learning

**Goal:** Train machine-learning models to predict whether a customer may leave a service (*churn*).

In this beginner-friendly project, we will:
- Generate a sample dataset locally (no external download required)
- Explore and prepare the data
- Split it into training and test sets
- Train Logistic Regression, Decision Tree, and Random Forest models
- Compare accuracy, precision, recall, F1-score, and ROC-AUC
- Visualize confusion matrices and ROC curves
- Explain the results in plain language

**Important:** This is synthetic demonstration data, not a real company's customer data. Model results are for learning, not for making real customer decisions.

## 1. Import libraries

We use `pandas` and `numpy` for data handling, `matplotlib` and `seaborn` for plots, and `scikit-learn` for machine learning.

In [ ]:
# If needed, uncomment and run:
# %pip install pandas numpy matplotlib seaborn scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score, classification_report
)

sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", None)
print("Libraries imported successfully.")

## 2. Create a sample customer dataset

Each row represents a customer. The features are illustrative:
- `tenure_months`: how long the customer has used the service
- `monthly_bill`: monthly bill amount
- `support_calls`: number of support calls
- `usage_hours`: approximate monthly usage
- `contract_length`: contract length in months
- `late_payments`: number of late payments
- `churn`: target label (1 = churn, 0 = stay)

The data is generated with a reproducible random seed so you can rerun the notebook and get the same sample.

In [ ]:
X_array, y = make_classification(
    n_samples=1000,
    n_features=6,
    n_informative=4,
    n_redundant=1,
    n_repeated=0,
    n_classes=2,
    weights=[0.70, 0.30],
    class_sep=0.9,
    flip_y=0.04,
    random_state=42
)

feature_names = [
    "tenure_months", "monthly_bill", "support_calls",
    "usage_hours", "contract_length", "late_payments"
]
df = pd.DataFrame(X_array, columns=feature_names)

# Convert standardized synthetic values into more intuitive, plausible ranges.
df["tenure_months"] = np.clip((df["tenure_months"] * 9 + 24).round(), 1, 72).astype(int)
df["monthly_bill"] = np.clip((df["monthly_bill"] * 18 + 80).round(2), 20, 250)
df["support_calls"] = np.clip((df["support_calls"] * 2 + 3).round(), 0, 20).astype(int)
df["usage_hours"] = np.clip((df["usage_hours"] * 12 + 35).round(1), 0, 120)
df["contract_length"] = np.where(df["contract_length"] > 0.5, 12, np.where(df["contract_length"] < -0.5, 1, 6))
df["late_payments"] = np.clip((df["late_payments"] * 1.5 + 2).round(), 0, 12).astype(int)
df["churn"] = y

# Add a few missing values to demonstrate preprocessing.
rng = np.random.default_rng(42)
for col in ["monthly_bill", "usage_hours", "support_calls"]:
    missing_idx = rng.choice(df.index, size=12, replace=False)
    df.loc[missing_idx, col] = np.nan

print("Dataset shape:", df.shape)
display(df.head(10))

## 3. Explore the data

Before modeling, check data types, descriptive statistics, missing values, and the distribution of the target label.

In [ ]:
print("Data types and non-null counts:")
df.info()

print("\nDescriptive statistics:")
display(df.describe().T)

print("\nMissing values:")
display(df.isna().sum().to_frame("Missing_Count"))

print("\nTarget balance (0 = stayed, 1 = churned):")
display(df["churn"].value_counts().rename(index={0: "Stayed", 1: "Churned"}).to_frame("Customers"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

df.isna().sum().sort_values(ascending=False).plot(kind="bar", ax=axes[0])
axes[0].set_title("Missing Values by Feature")
axes[0].set_xlabel("Feature")
axes[0].set_ylabel("Missing cells")
axes[0].tick_params(axis="x", rotation=35)

sns.countplot(data=df, x="churn", ax=axes[1])
axes[1].set_title("Target Class Distribution")
axes[1].set_xlabel("Churn (0 = stayed, 1 = churned)")
axes[1].set_ylabel("Number of customers")

plt.tight_layout()
plt.show()

## 4. Prepare features and target

`X` contains the input features; `y` contains the outcome we want to predict. We keep a test set separate from training so evaluation reflects performance on data the model did not learn from.

The preprocessing pipeline fills missing numeric values with the median. Logistic Regression also needs feature scaling, so we include a scaler in that model's pipeline. Tree-based models do not generally require scaling.

In [ ]:
X = df.drop(columns="churn")
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("\nTraining target proportions:")
display(y_train.value_counts(normalize=True).rename("proportion").to_frame())

## 5. Train three classification models

- **Logistic Regression:** a simple, useful baseline for binary classification.
- **Decision Tree:** learns rule-like splits in the data; easy to explain but can overfit.
- **Random Forest:** combines many decision trees, often improving generalization.

We use pipelines so missing-value imputation and scaling are fitted using the training data only.

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, random_state=42))
    ]),
    "Decision Tree": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DecisionTreeClassifier(max_depth=5, random_state=42))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(n_estimators=250, max_depth=8, random_state=42))
    ])
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"Trained: {name}")

## 6. Evaluate performance

Accuracy is the fraction of predictions that are correct. Precision answers “of the customers predicted to churn, how many actually churned?” Recall answers “of all customers who churned, how many did the model identify?” F1 balances precision and recall.

For churn detection, recall can be especially important if the goal is to identify as many at-risk customers as possible. The best metric depends on the real business objective.

In [ ]:
results = []
predictions = {}
probabilities = {}

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    predictions[name] = y_pred
    probabilities[name] = y_prob

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1-score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results).set_index("Model").sort_values("F1-score", ascending=False)
display(results_df.round(3))

In [ ]:
# Compare the main metrics visually
results_long = results_df.reset_index().melt(
    id_vars="Model", var_name="Metric", value_name="Score"
)

plt.figure(figsize=(12, 5.5))
sns.barplot(data=results_long, x="Metric", y="Score", hue="Model")
plt.ylim(0, 1.05)
plt.title("Model Performance Comparison")
plt.ylabel("Score (higher is better)")
plt.xlabel("Evaluation metric")
plt.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 7. Confusion matrices

A confusion matrix shows:
- **True negative (TN):** correctly predicted to stay
- **False positive (FP):** predicted to churn, but stayed
- **False negative (FN):** predicted to stay, but churned
- **True positive (TP):** correctly predicted to churn

False negatives can matter if the business wants to reach customers before they leave.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for ax, (name, y_pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Stayed (0)", "Churned (1)"])
    disp.plot(ax=ax, colorbar=False, values_format="d")
    ax.set_title(name)

plt.tight_layout()
plt.show()

## 8. ROC curves and AUC

The ROC curve compares the true-positive rate with the false-positive rate across possible classification thresholds. AUC summarizes the curve: 0.5 is roughly chance-level ranking, while 1.0 is perfect ranking. AUC does not tell us whether the chosen prediction threshold is appropriate.

In [ ]:
plt.figure(figsize=(8, 6))

for name, y_prob in probabilities.items():
    fpr, tpr, thresholds = roc_curve(y_test, y_prob)
    auc_value = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc_value:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", label="Random baseline")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC Curves — Model Comparison")
plt.legend()
plt.tight_layout()
plt.show()

## 9. Inspect the best model's classification report

We select the model with the highest F1-score on this held-out test set for demonstration. In a serious project, choose metrics and tune models using cross-validation on training data, then evaluate once on the test set.

In [ ]:
best_model_name = results_df.index[0]
best_model = models[best_model_name]
best_predictions = predictions[best_model_name]

print("Selected model by test-set F1-score:", best_model_name)
print()
print(classification_report(
    y_test, best_predictions,
    target_names=["Stayed", "Churned"],
    zero_division=0
))

## 10. Understand feature importance (Random Forest)

Random Forest can provide a rough measure of how useful each feature was for splitting the data. This is not proof that a feature causes churn, and importance can be affected by feature relationships.

In [ ]:
rf_pipeline = models["Random Forest"]
rf_model = rf_pipeline.named_steps["model"]
importance = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(x=importance.values, y=importance.index)
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

display(importance.rename("Importance").to_frame().round(3))

## 11. Tell the story from the results

The following summary uses the actual scores calculated above. Since the data is synthetic, these findings demonstrate how to report model results; they should not be interpreted as real-world churn evidence.

In [ ]:
best_row = results_df.iloc[0]
print("MODEL EVALUATION SUMMARY")
print("-" * 55)
print(f"Best model by F1-score: {best_model_name}")
print(f"Accuracy:  {best_row['Accuracy']:.3f}")
print(f"Precision: {best_row['Precision']:.3f}")
print(f"Recall:    {best_row['Recall']:.3f}")
print(f"F1-score:  {best_row['F1-score']:.3f}")
print(f"ROC-AUC:   {best_row['ROC-AUC']:.3f}")

print("\nHow to interpret this:")
print("- Accuracy shows the overall share of correct predictions.")
print("- Precision indicates how often a churn alert was correct.")
print("- Recall indicates how many churn cases were identified.")
print("- F1-score balances precision and recall.")
print("- ROC-AUC indicates how well the model ranks churn cases above non-churn cases.")

print("\nBefore deployment, validate on real representative data, check for bias,")
print("choose a threshold based on business costs, and monitor performance over time.")

## 12. Try a prediction for a new customer

This final example demonstrates how to pass one new row through the selected pipeline. Values are illustrative. The output is a model prediction, not a certainty.

In [ ]:
new_customer = pd.DataFrame([{
    "tenure_months": 8,
    "monthly_bill": 145.0,
    "support_calls": 7,
    "usage_hours": 18.0,
    "contract_length": 1,
    "late_payments": 4
}])

predicted_class = int(best_model.predict(new_customer)[0])
churn_probability = float(best_model.predict_proba(new_customer)[0, 1])

print("Predicted class:", "Churn" if predicted_class == 1 else "Stay")
print(f"Estimated churn probability: {churn_probability:.1%}")

## Conclusion

In this project, you built and compared three supervised classification algorithms, used a held-out test set, and evaluated models with classification metrics, confusion matrices, and ROC curves.

**Key lesson:** no single metric tells the whole story. Choose the model and decision threshold based on the cost of missed churn cases versus unnecessary retention outreach. Keep data preparation inside pipelines to reduce leakage, and never treat predictions as guaranteed outcomes.